# ThemeCloner V2 — revised point in time walk forward

This notebook keeps the existing data pull and RP PCA modules intact. The revised logic is isolated in three additive modules:

- `src/matching_v3.py`: robust multi ETF factor exposure matching
- `src/walkforward_v3.py`: point in time residualization, RP PCA fitting and basket formation
- `src/evaluation_v3.py`: forward exposure, placebo and semantic validation

The primary objective is to identify small cap stocks with related latent thematic exposures. Raw ETF return tracking is retained only as a secondary diagnostic.

In [1]:
# -------------------- 0. Setup --------------------
import os
import sys

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, "config", "etfs.csv")):
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print(f"root: {ROOT}")
print(f"config exists: {os.path.exists('config/etfs.csv')}")

root: C:\Users\aamin\ThemeCloner2
config exists: True


## Step 1 — Pull the same raw inputs used by V2

In [2]:
# -------------------- 1a. Parameters and ETF config --------------------
from src.data_pull import load_etf_config, split_themes_controls

START_DATE = "2018-01-01"
K = 15
GAMMA = 10.0
TOP_N = 30
TOP_FACTORS = 5                      # retain more of each theme's factor shape

REBALANCE_FREQ = "Q"
WALKFORWARD_WINDOW = "rolling"      # "rolling" or "expanding"
ROLLING_WINDOW_WEEKS = 208          # four years
MIN_TRAIN_OBS = 104                 # at least two years of valid observations
MIN_TRAIN_COVERAGE = 0.80

INCLUDE_COMMODITY = False
N_PLACEBOS = 20                     # use 100 to 200 for final reported results
PLACEBO_PVALUE_MAX = 0.10           # rank matched null admission threshold
PLACEBO_USE_FDR = False             # q values are reported; p values gate admission
RANDOM_STATE = 42

# Process workers are used for placebo batches. Half the logical CPUs avoids
# oversubscription on mixed performance and efficiency core laptops.
N_JOBS = min(8, max(1, (os.cpu_count() or 4) // 2))
print(f"Placebo process workers: {N_JOBS}")

etf_config = load_etf_config("config/etfs.csv")
themes_config, controls_config = split_themes_controls(etf_config)
print(f"{themes_config['theme'].nunique()} discovery themes")


Placebo process workers: 8
loaded 44 ETFs: 31 theme-ETFs, 13 control-ETFs (sector/subsector calibration anchors)
  AI Infrastructure: IGPT (IGPT), QTUM (QTUM), SOXX (SOXX)
  Agribusiness: MOO (MOO), VEGI (VEGI), PBJ (PBJ)
  Clean Energy: ICLN (ICLN), QCLN (QCLN), PBW (PBW)
  Critical Minerals: URA (URA), LIT (LIT), COPX (COPX)
  Cybersecurity: CIBR (CIBR), HACK (HACK), BUG (BUG)
  Defense: ITA (ITA), PPA (PPA), XAR (XAR)
  Factor1: IUSG (IUSG)
  Factor2: VLUE (VLUE)
  Factor3: QUAL (QUAL)
  Factor4: SPGP (SPGP)
  Factor5: MTUM (MTUM)
  Factor6: VUG (VUG)
  Factor7: VTV (VTV)
  Fintech: FINX (FINX), ARKF (ARKF), IPAY (IPAY)
  Robotics & AI: BOTZ (BOTZ), ROBO (ROBO), IRBO (IRBO)
  SECTOR: US Energy: XLE (XLE)
  SECTOR: US Financials: XLF (XLF)
  SECTOR: US Technology: XLK (XLK)
  SUBSECTOR: US Banks: KRE (KRE)
  SUBSECTOR: US Biotech: XBI (XBI)
  SUBSECTOR: US Software: IGV (IGV)
  Timber & Forestry: WOOD (WOOD), CUT (CUT)
  US Infrastructure: PAVE (PAVE), IFRA (IFRA)
  Water: PHO (PHO),

In [3]:
# -------------------- 1b. ETF returns --------------------
from src.data_pull import pull_etf_returns

etf_returns = pull_etf_returns(etf_config, start=START_DATE)
print(etf_returns.shape)


pulling ETF panel: 44 tickers in 1 batch(es)
  batch 1/1 (44 tickers)...
  dropped 1 tickers below 85% coverage
  ETF panel: 445 weeks x 43 stocks
  date range: 2018-01-12 to 2026-07-17
  ETFs kept: ['ARKF', 'BOTZ', 'CGW', 'CIBR', 'COPX', 'CUT', 'FINX', 'FIW', 'HACK', 'ICLN', 'IFRA', 'IGPT', 'IGV', 'IPAY', 'IRBO', 'ITA', 'IUSG', 'KRE', 'LIT', 'MOO', 'MTUM', 'PAVE', 'PBJ', 'PBW', 'PHO', 'PPA', 'QCLN', 'QTUM', 'QUAL', 'ROBO', 'SOXX', 'SPGP', 'URA', 'VEGI', 'VLUE', 'VTV', 'VUG', 'WOOD', 'XAR', 'XBI', 'XLE', 'XLF', 'XLK']
(445, 43)


In [4]:
# -------------------- 1c. Covariance universe --------------------
from src.data_pull import pull_covariance_universe

cov_returns = pull_covariance_universe(
    start=START_DATE,
    use_cache=True,
    us_only=True,
)
print(cov_returns.shape)

loaded covariance universe from cache (us_only): 1003 tickers across ['us_sp500', 'us_mid_small']

pulling covariance universe: 1003 unique tickers

pulling covariance universe: 1003 tickers in 3 batch(es)
  batch 1/3 (400 tickers)...


$INVH: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)
$SATS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

2 Failed downloads:
['INVH', 'SATS']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 2/3 (400 tickers)...


$BAYA: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['BAYA']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 3/3 (203 tickers)...


$TBH: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['TBH']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  dropped 252 tickers below 85% coverage
  covariance universe: 445 weeks x 751 stocks
  date range: 2018-01-12 to 2026-07-17
(445, 751)


In [5]:
# -------------------- 1d. Target universe --------------------
from src.data_pull import _cached_returns, pull_target_universe

tgt_returns = _cached_returns(
    "target_universe_returns",
    pull_target_universe,
    refresh=False,
    start=START_DATE,
)
print(tgt_returns.shape)

  loaded target_universe_returns from cache: 444 weeks x 811 cols
(444, 811)


In [6]:
# -------------------- 1e. Observable factor returns --------------------
# IMPORTANT: do not residualize the full sample here.
# The revised walk forward module fits residualization inside each rebalance window.
from src.residualize import get_combined_factors

ff_factors = get_combined_factors(
    start=START_DATE,
    include_commodity=INCLUDE_COMMODITY,
)
print(ff_factors.shape)


pulling FF5 + momentum from Kenneth French library (weekly)...
  momentum fetch failed via pandas_datareader (Unknown datetime string format, unable to parse: Missing data are indicated by -99.99 or -999., at position 0)
  falling back to direct download from Ken French's data library...
  first 20 raw lines (for diagnosis):
    'This file was created by using the 202605 CRSP database.  It,,'
    'contains a momentum factor, constructed from six value-weight portfolios formed,'
    'using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks.'
    'MOM is the average of the returns on two (big and small) high prior return portfolios,,'
    'minus the average of the returns on two low prior return portfolios.  The portfolios,,'
    'are constructed daily.  Big means a firm is above the median market cap on the NYSE,,'
    'at the end of the previous day; small firms are below the median NYSE market cap.,,'
    'Prior return is measured from day - 250 to - 21.  Fir

## Step 2 — Build the point in time rebalance schedule

In [7]:
# -------------------- 2. Rebalance dates --------------------
from src.rppca_walkforward import make_rebalance_dates

# Restrict the schedule to dates where the observable factor panel is available.
model_dates = cov_returns.index.intersection(ff_factors.index)
minimum_window = ROLLING_WINDOW_WEEKS if WALKFORWARD_WINDOW == "rolling" else MIN_TRAIN_OBS

rebalance_dates = make_rebalance_dates(
    model_dates,
    freq=REBALANCE_FREQ,
    min_window_weeks=minimum_window,
)
print(f"{len(rebalance_dates)} rebalance dates")
print(rebalance_dates[:3], "...", rebalance_dates[-3:])

18 rebalance dates
[Timestamp('2021-12-31 00:00:00'), Timestamp('2022-03-25 00:00:00'), Timestamp('2022-06-24 00:00:00')] ... [Timestamp('2025-09-26 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2026-03-27 00:00:00')]


## Step 3 — Matching rules

In [8]:
# -------------------- 3. Robust matching configuration --------------------
from src.matching_v3 import MatchingConfig

matching_config = MatchingConfig(
    top_factors=TOP_FACTORS,

    # Confidence filters use both raw and adjusted R squared.
    min_etf_r2=0.10,
    min_etf_adjusted_r2=0.05,
    min_candidate_r2=0.15,
    min_candidate_adjusted_r2=0.10,

    # Consensus checks across the theme ETF reference set.
    min_cosine=0.30,
    cosine_quantile=0.25,            # lower quartile cosine must clear the floor
    reference_distance_quantile=0.75,# poor ETF match cannot be averaged away

    max_relative_distance=None,
    factor_gap_weight=0.25,
    consensus_weight=0.25,

    min_etf_matches=1,
    min_etf_match_fraction=0.60,     # 2 of 2 ETFs; at least 2 of 3 ETFs
    etf_match_distance=None,

    # Overlap is diagnosed, not prohibited.
    max_theme_rank=None,
)
matching_config


MatchingConfig(top_factors=5, min_etf_r2=0.1, min_etf_adjusted_r2=0.05, min_candidate_r2=0.15, min_candidate_adjusted_r2=0.1, min_cosine=0.3, cosine_quantile=0.25, reference_distance_quantile=0.75, max_relative_distance=None, factor_gap_weight=0.25, consensus_weight=0.25, min_etf_matches=1, min_etf_match_fraction=0.6, etf_match_distance=None, max_theme_rank=None)

## Step 4 — Run the revised walk forward process

### 3B. Parallel execution

Ticker regressions are vectorized. Independent placebo batches use separate process workers, so the earlier Windows `threadpoolctl` workaround is no longer required.


In [9]:
# -------------------- 4. Point in time backtest --------------------
from src.walkforward_v3 import run_point_in_time_backtest

results = run_point_in_time_backtest(
    cov_returns_raw=cov_returns,
    etf_returns_raw=etf_returns,
    target_returns_raw=tgt_returns,
    factor_returns=ff_factors,
    etf_config=themes_config,
    rebalance_dates=rebalance_dates,
    K=K,
    gamma=GAMMA,
    window=WALKFORWARD_WINDOW,
    rolling_window_weeks=ROLLING_WINDOW_WEEKS,
    top_n=TOP_N,
    matching_config=matching_config,
    min_train_obs=MIN_TRAIN_OBS,
    min_train_coverage=MIN_TRAIN_COVERAGE,
    n_placebos=N_PLACEBOS,
    placebo_pvalue_max=PLACEBO_PVALUE_MAX,
    placebo_use_fdr=PLACEBO_USE_FDR,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=True,
)



Rebalance 1/18: 2021-12-31
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.052
  top eigenvalues:   [61.4 41.5 25.2 21.2 17.4 16.7 14.9 13.  12.5 11.4 10.2  9.3  9.2  8.8
  8.7]
Selected 170 positions across 11 themes; forward window has 12 observations.
Timing seconds: residualization=0.3, rppca=1.9, reference_and_candidate_fit=6.6, placebos_and_admission=123.0, forward_evaluation=2.7, placebo_forward_metrics=1.6, total=136.1

Rebalance 2/18: 2022-03-25
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.538
  top eigenvalues:   [63.3 40.4 25.4 22.6 18.3 16.3 14.7 13.5 12.5 11.4 10.4  9.6  9.   8.9
  8.7]
Selected 187 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.9, rppca=0.9, reference_and_candidate_fit=6.4, placebos_and_admission=94.7, forward_evaluation=0.9, placebo_forward_metrics=0.6, total=104.3

Rebalance 3/18: 2022-06-24
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.451
  top eigenvalues:   [62.4 35.6 26.3

RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.662
  top eigenvalues:   [62.1 49.4 26.3 20.6 19.7 15.4 14.8 13.6 11.5 11.  10.2  9.5  9.2  9.2
  8.8]
Selected 126 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.2, rppca=0.9, reference_and_candidate_fit=5.8, placebos_and_admission=22.3, forward_evaluation=0.6, placebo_forward_metrics=0.4, total=30.2

Rebalance 16/18: 2025-09-26
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.028
  top eigenvalues:   [64.3 47.5 25.9 20.3 19.  16.5 14.1 13.7 11.5 11.2 10.2  9.7  9.3  8.9
  8.7]
Selected 142 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.2, rppca=0.9, reference_and_candidate_fit=6.1, placebos_and_admission=17.3, forward_evaluation=0.6, placebo_forward_metrics=0.3, total=25.2

Rebalance 17/18: 2025-12-26
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.298
  top eigenvalues:   [67.4 46.  26.5 20.3 18.8 16.9 14.3 12.7 12.

## Step 5 — Review the latest selected baskets

In [10]:
# -------------------- 5a. Latest baskets --------------------
from src.matching_v3 import flatten_baskets

basket_history = flatten_baskets(results["period_baskets"])

if basket_history.empty:
    print(
        "No stocks passed the current matching thresholds in any rebalance period."
    )

    # Show how many names passed in each period and theme.
    selection_counts = pd.DataFrame(
        [
            {
                "rebalance_date": date,
                "theme": theme,
                "n_selected": len(basket.get("tickers", [])),
            }
            for date, theme_map in results["period_baskets"].items()
            for theme, basket in theme_map.items()
        ]
    )

    display(selection_counts)
else:
    latest_date = basket_history["rebalance_date"].max()
    latest_baskets = basket_history.loc[
        basket_history["rebalance_date"] == latest_date
    ]

    print(f"Latest rebalance: {latest_date.date()}")
    display(
        latest_baskets.sort_values(["theme", "ticker"])
    )

Latest rebalance: 2026-03-27


,rebalance_date,theme,ticker,weight
2773,2026-03-27,AI Infrastructure,NVEC,1.000000
2776,2026-03-27,Agribusiness,ACNT,0.090909
2778,2026-03-27,Agribusiness,ASA,0.090909
2782,2026-03-27,Agribusiness,CVGI,0.090909
2780,2026-03-27,Agribusiness,GOOS,0.090909
...,...,...,...,...
2924,2026-03-27,US Infrastructure,MTUS,0.333333
2926,2026-03-27,US Infrastructure,NGVC,0.333333
2925,2026-03-27,US Infrastructure,SXC,0.333333
2928,2026-03-27,Water,GWRS,0.500000


In [11]:
# -------------------- 5c. Rebuild baskets without placebo admission --------------------
from src.matching_v3 import select_equal_weight_baskets, flatten_baskets

period_baskets_base = {}

for date, scores in results["period_scores"].items():
    revised_scores = scores.copy()
    revised_scores["eligible"] = revised_scores["base_eligible"]

    period_baskets_base[date] = select_equal_weight_baskets(
        revised_scores,
        top_n=TOP_N,
    )

basket_history_base = flatten_baskets(period_baskets_base)

latest_date = basket_history_base["rebalance_date"].max()
latest_baskets_base = basket_history_base.loc[
    basket_history_base["rebalance_date"] == latest_date
]

print(f"Latest rebalance: {latest_date.date()}")
print(f"Selected positions: {len(latest_baskets_base)}")
print(f"Unique stocks: {latest_baskets_base['ticker'].nunique()}")

display(
    latest_baskets_base.sort_values(["theme", "ticker"])
)

Latest rebalance: 2026-03-27
Selected positions: 156
Unique stocks: 98


,rebalance_date,theme,ticker,weight
2773,2026-03-27,AI Infrastructure,NVEC,1.000000
2776,2026-03-27,Agribusiness,ACNT,0.090909
2778,2026-03-27,Agribusiness,ASA,0.090909
2782,2026-03-27,Agribusiness,CVGI,0.090909
2780,2026-03-27,Agribusiness,GOOS,0.090909
...,...,...,...,...
2924,2026-03-27,US Infrastructure,MTUS,0.333333
2926,2026-03-27,US Infrastructure,NGVC,0.333333
2925,2026-03-27,US Infrastructure,SXC,0.333333
2928,2026-03-27,Water,GWRS,0.500000


In [12]:
# -------------------- 5b. Latest detailed scores --------------------

latest_scores = results["period_scores"][latest_date]

latest_scores.loc[
    latest_scores["base_eligible"]
].sort_values(
    ["theme", "penalized_distance"]
).groupby(
    "theme",
    sort=False,
).head(10)[
    [
        "theme",
        "ticker",
        "penalized_distance",
        "consensus_cosine",
        "candidate_r2",
        "candidate_adjusted_r2",
        "n_etf_matches",
        "required_etf_matches",
        "placebo_rank_pvalue",
        "theme_distance_rank",
        "n_eligible_themes",
    ]
]

,theme,ticker,penalized_distance,consensus_cosine,candidate_r2,candidate_adjusted_r2,n_etf_matches,required_etf_matches,placebo_rank_pvalue,theme_distance_rank,n_eligible_themes
0,AI Infrastructure,NVEC,1.958285,0.418210,0.173763,0.109213,2,2,0.500000,3.0,3
811,Agribusiness,PFX,1.342896,0.384090,0.168323,0.103348,3,2,0.142857,2.0,4
812,Agribusiness,HRZN,1.952650,0.310947,0.166556,0.101443,2,2,0.142857,5.0,2
813,Agribusiness,ACNT,2.354858,0.419372,0.165801,0.100629,3,2,0.142857,3.0,3
814,Agribusiness,MTUS,2.397136,0.313117,0.195026,0.132137,2,2,0.142857,6.0,4
...,...,...,...,...,...,...,...,...,...,...,...
7299,US Infrastructure,MTUS,2.427423,0.385001,0.195026,0.132137,2,2,0.200000,7.0,4
7300,US Infrastructure,SXC,2.692692,0.408693,0.256534,0.198451,2,2,0.200000,5.0,3
7301,US Infrastructure,NGVC,3.353016,0.310264,0.219713,0.158753,2,2,0.400000,4.0,2
8110,Water,PFX,1.484733,0.415660,0.168323,0.103348,3,2,0.166667,4.0,4


In [13]:
# -------------------- 5c. Selection overlap diagnostic --------------------

theme_counts = (
    latest_baskets_base.groupby("theme")
    .agg(
        n_selected=("ticker", "size"),
        n_unique=("ticker", "nunique"),
    )
    .sort_values("n_selected")
)

ticker_overlap = (
    latest_baskets_base.groupby("ticker")
    .agg(
        n_themes=("theme", "nunique"),
        themes=("theme", lambda x: ", ".join(sorted(set(x)))),
    )
    .query("n_themes > 1")
    .sort_values(["n_themes", "ticker"], ascending=[False, True])
)

display(theme_counts)
display(ticker_overlap.head(30))

,n_selected,n_unique
theme,,
AI Infrastructure,1,1
Water,2,2
US Infrastructure,3,3
Agribusiness,11,11
Defense,11,11
Fintech,12,12
Clean Energy,14,14
Robotics & AI,16,16
Critical Minerals,28,28


,n_themes,themes
ticker,,
GOOS,5,"Agribusiness, Clean Energy, Critical Minerals,..."
ASA,4,"Agribusiness, Critical Minerals, Defense, Timb..."
CVGI,4,"Agribusiness, Clean Energy, Critical Minerals,..."
MTUS,4,"Agribusiness, Critical Minerals, Timber & Fore..."
PFX,4,"Agribusiness, Defense, Robotics & AI, Water"
ACNT,3,"Agribusiness, Critical Minerals, Timber & Fore..."
CSIQ,3,"Clean Energy, Critical Minerals, Robotics & AI"
ELME,3,"Cybersecurity, Fintech, Timber & Forestry"
GSM,3,"Agribusiness, Critical Minerals, Timber & Fore..."


In [14]:
# -------------------- 5d. Are the theme reference sets distinct? --------------------
latest_reference_distances = results["theme_reference_distances"]
latest_reference_distances = latest_reference_distances.loc[
    latest_reference_distances["rebalance_date"] == latest_date
]
latest_reference_distances.sort_values(
    ["median_relative_distance", "maximum_cosine"],
    ascending=[True, False],
).head(30)


,rebalance_date,theme_1,theme_2,median_relative_distance,minimum_relative_distance,maximum_cosine,median_cosine
959,2026-03-27,Clean Energy,Critical Minerals,0.854807,0.668145,0.639305,0.478787
983,2026-03-27,US Infrastructure,Timber & Forestry,0.855808,0.686124,0.567596,0.439635
982,2026-03-27,US Infrastructure,Water,0.882917,0.548401,0.720658,0.464912
989,2026-03-27,Water,Timber & Forestry,0.887392,0.746426,0.392626,0.270335
945,2026-03-27,AI Infrastructure,Clean Energy,0.904742,0.773363,0.542761,0.431262
935,2026-03-27,Robotics & AI,AI Infrastructure,0.909275,0.714739,0.719795,0.618277
986,2026-03-27,Agribusiness,Timber & Forestry,0.968804,0.712878,0.690804,0.510201
988,2026-03-27,Critical Minerals,Timber & Forestry,0.985500,0.708951,0.737918,0.684536
985,2026-03-27,Agribusiness,Water,1.005948,0.825407,0.429082,0.167820
970,2026-03-27,Defense,US Infrastructure,1.031488,0.970541,0.476802,0.344389


In [15]:
# -------------------- 5e. Runtime bottleneck diagnostic --------------------
results["timings"].sort_values("rebalance_date")


,rebalance_date,residualization,rppca,reference_and_candidate_fit,placebos_and_admission,forward_evaluation,placebo_forward_metrics,total
0,2021-12-31,0.261957,1.891210,6.609999,123.022242,2.668226,1.616652,136.070300
1,2022-03-25,0.862073,0.914620,6.399470,94.675711,0.862925,0.577181,104.291996
2,2022-06-24,0.271497,1.038898,6.538637,70.562061,2.403347,1.354729,82.169193
3,2022-09-30,0.411370,3.217352,12.851726,61.594582,1.445787,0.696912,80.217753
4,2022-12-30,0.354738,1.926935,8.428473,50.385138,0.851106,0.521602,62.468566
5,2023-03-31,0.191751,0.732623,6.747238,93.066725,1.348366,1.190077,103.276795
6,2023-06-30,0.415385,1.295943,13.426657,106.052160,1.210448,1.408733,123.809342
7,2023-09-29,0.252783,1.868416,9.095488,99.674698,1.294703,2.645207,114.831313
8,2023-12-29,0.448035,1.420048,9.320019,74.029870,1.613952,0.797405,87.629346
9,2024-03-29,0.210692,1.114956,9.528186,63.904964,1.092283,0.977876,76.828971


## Step 6 — Forward exposure evaluation

In [16]:
# -------------------- 6a. Theme level summary --------------------
from src.evaluation_v3 import summarize_forward_evaluation

evaluation_summary = summarize_forward_evaluation(results["evaluation"])
evaluation_summary.sort_values("median_rank_ic", ascending=False)

,theme,periods,median_rank_ic,mean_rank_ic,median_top_bottom_spread,median_selected_forward_corr,median_basket_etf_corr,median_exposure_hit_rate,median_placebo_pvalue,rank_ic_positive_rate
9,Water,18,0.338811,0.203693,0.165663,0.132952,0.318476,0.633333,0.550000,0.700000
3,Cybersecurity,18,0.173182,0.160538,0.156661,0.083882,0.248154,0.634848,0.500000,0.777778
10,Timber & Forestry,18,0.096698,0.046499,0.116649,0.089060,0.035751,0.599673,0.500000,0.611111
2,Clean Energy,18,0.094260,0.067960,0.134867,0.103823,0.179931,0.650000,0.500000,0.611111
4,Defense,18,0.005986,0.015059,-0.022996,0.117441,0.233254,0.634848,0.500000,0.500000
7,Agribusiness,18,-0.032555,-0.013092,-0.137356,0.166653,0.186737,0.708333,0.333333,0.500000
1,AI Infrastructure,17,-0.050000,-0.017929,-0.136508,0.099600,0.332798,0.750000,0.416667,0.333333
0,Robotics & AI,18,-0.069143,-0.060467,-0.059112,0.091121,0.191562,0.645768,0.500000,0.333333
8,Critical Minerals,18,-0.079130,-0.143920,-0.091692,0.230937,0.479300,0.816667,0.250000,0.333333
5,Fintech,18,-0.079697,0.043147,0.009740,0.155115,0.159900,0.666667,0.416667,0.437500


In [17]:
# -------------------- 6b. Period level metrics --------------------
# Primary metrics:
# 1. forward_rank_ic_corr
# 2. top_bottom_corr_spread
# 3. basket_residual_etf_corr
# 4. placebo_pvalue
results["evaluation"].sort_values(["theme", "rebalance_date"])

,rebalance_date,theme,n_selected,n_base_eligible,forward_rank_ic_beta,forward_rank_ic_corr,top_bottom_beta_spread,top_bottom_corr_spread,selected_mean_forward_beta,selected_mean_forward_corr,exposure_hit_rate,basket_synthetic_beta,basket_synthetic_corr,basket_residual_etf_beta,basket_residual_etf_corr,raw_basket_period_return,n_benchmark_etfs,placebo_pvalue,placebo_median_basket_corr
1,2021-12-31,AI Infrastructure,8,8,-0.095238,0.00000,NaN,NaN,3.157465,0.318072,0.75,3.157465,0.780784,0.583509,0.378090,0.018429,3,0.166667,-0.223359
12,2022-03-25,AI Infrastructure,10,10,-0.054545,0.10303,-0.539517,-0.040188,1.672976,0.161831,0.80,1.672976,0.373687,0.907980,0.442236,-0.065729,3,0.250000,-0.237920
23,2022-06-24,AI Infrastructure,5,5,-0.200000,-0.10000,NaN,NaN,0.087929,0.020286,0.60,0.087929,0.026124,0.514874,0.333785,-0.120948,3,0.750000,0.103617
34,2022-09-30,AI Infrastructure,5,5,-0.300000,-0.10000,NaN,NaN,0.882294,0.082625,0.60,0.882294,0.255702,0.713645,0.380348,-0.088895,3,0.333333,-0.278621
45,2022-12-30,AI Infrastructure,1,1,NaN,NaN,NaN,NaN,5.322725,0.384676,1.00,5.322725,0.384676,3.181437,0.253118,-0.045901,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,2025-03-28,Water,2,2,NaN,NaN,NaN,NaN,0.124167,0.135853,0.50,0.124167,0.036714,0.425813,0.169107,-0.100036,3,0.800000,0.174455
161,2025-06-27,Water,4,4,NaN,NaN,NaN,NaN,-0.375223,-0.084126,0.25,-0.375223,-0.116677,-0.444322,-0.271539,-0.090320,3,0.500000,-0.174429
171,2025-09-26,Water,2,2,NaN,NaN,NaN,NaN,1.077656,0.212105,1.00,1.077656,0.289767,0.778478,0.452215,-0.077391,3,0.333333,-0.172907
182,2025-12-26,Water,4,4,NaN,NaN,NaN,NaN,1.373419,0.431031,1.00,1.373419,0.746615,0.682331,0.397356,-0.013759,3,0.500000,-0.346665


In [18]:
# -------------------- 6c. Rank matched placebo admission --------------------
# Each real rank is compared with the same rank from time shifted null themes.
results["placebo_rank_thresholds"].tail(30)


,rebalance_date,theme,rank,actual_penalized_distance,null_alpha_distance,placebo_rank_pvalue,n_placebo_ranks
2305,2026-03-27,Timber & Forestry,6,2.046288,2.594808,0.500000,1
2306,2026-03-27,Timber & Forestry,7,2.067433,2.682142,0.500000,1
2307,2026-03-27,Timber & Forestry,8,2.124601,2.820303,0.500000,1
2308,2026-03-27,Timber & Forestry,9,2.129378,3.105684,0.500000,1
2309,2026-03-27,Timber & Forestry,10,2.210393,3.126995,0.500000,1
2310,2026-03-27,Timber & Forestry,11,2.254371,3.235527,0.500000,1
2311,2026-03-27,Timber & Forestry,12,2.258206,3.288658,0.500000,1
2312,2026-03-27,Timber & Forestry,13,2.438961,3.304428,0.500000,1
2313,2026-03-27,Timber & Forestry,14,2.452802,3.360461,0.500000,1
2314,2026-03-27,Timber & Forestry,15,2.500782,3.403525,0.500000,1


## Step 7 — Independent economic relevance review

In [19]:
# -------------------- 7a. Create a blinded review sample --------------------
# Score each company from 0 to 3 using filings available before the rebalance date:
# 0 = no exposure, 1 = incidental, 2 = meaningful, 3 = central business driver.
import importlib
import src.evaluation_v3 as ev3


importlib.reload(ev3)
make_semantic_review_sample = ev3.make_semantic_review_sample

semantic_sample = make_semantic_review_sample(
    latest_scores,
    metadata=None,          # optionally pass ticker indexed sector and market cap data
    top_k=10,
    controls_per_candidate=1,
    random_state=RANDOM_STATE,
)

os.makedirs("outputs", exist_ok=True)
semantic_sample["review_sheet"].to_csv(
    "outputs/semantic_review_sheet.csv", index=False
)
semantic_sample["review_key"].to_csv(
    "outputs/semantic_review_key_PRIVATE.csv", index=False
)
semantic_sample["review_sheet"].head()

,review_id,theme,ticker,relevance_score,reviewer_notes
0,R0077,Cybersecurity,WTBA,NaN,
1,R0139,Robotics & AI,EDRY,NaN,
2,R0153,Timber & Forestry,ASIX,NaN,
3,R0061,Critical Minerals,MRAM,NaN,
4,R0157,Timber & Forestry,XRN,NaN,


In [20]:
# -------------------- 7b. Evaluate a completed review --------------------
# Run only after filling relevance_score in outputs/semantic_review_sheet.csv.
from src.evaluation_v3 import evaluate_semantic_review

completed_review = pd.read_csv("outputs/semantic_review_sheet.csv")
review_key = pd.read_csv("outputs/semantic_review_key_PRIVATE.csv")

n_completed = pd.to_numeric(
    completed_review["relevance_score"], errors="coerce"
).notna().sum()

if n_completed == 0:
    print(
        "No semantic reviews completed. "
        "Fill relevance_score before running this evaluation."
    )
else:
    semantic_results = evaluate_semantic_review(completed_review, review_key)
    display(semantic_results)


No semantic reviews completed. Fill relevance_score before running this evaluation.


## Interpretation

A credible theme should show positive forward rank correlation, a positive top versus bottom exposure spread, positive basket correlation to the residualized ETF blend, and a low placebo p value. The semantic review is a separate economic check. Raw basket return remains secondary because the objective is exposure discovery rather than replication of a large cap ETF.

In [21]:
# -------------------- 8. Export run results for review --------------------
from pathlib import Path
import zipfile

repo = Path.cwd()
outputs_dir = repo / "outputs"
outputs_dir.mkdir(exist_ok=True)

# Export the new diagnostics automatically.
results["evaluation"].to_csv(outputs_dir / "v3_forward_evaluation.csv", index=False)
results["candidate_forward_exposure"].to_csv(
    outputs_dir / "v3_candidate_forward_exposure.csv",
    index=False,
)
results["placebo_rank_thresholds"].to_csv(
    outputs_dir / "v3_placebo_rank_thresholds.csv",
    index=False,
)
results["theme_reference_distances"].to_csv(
    outputs_dir / "v3_theme_reference_distances.csv",
    index=False,
)
results["overlap_diagnostics"].to_csv(
    outputs_dir / "v3_overlap_diagnostics.csv",
    index=False,
)
results["timings"].to_csv(outputs_dir / "v3_timings.csv", index=False)
basket_history.to_csv(outputs_dir / "v3_basket_history.csv", index=False)

zip_path = repo / "ThemeCloner_V3_run_results.zip"
files_to_include = [
    repo / "ThemeCloner_V2_WalkForward_revised.ipynb",
    repo / "src" / "matching_v3.py",
    repo / "src" / "walkforward_v3.py",
    repo / "src" / "evaluation_v3.py",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_include:
        if file.exists():
            z.write(file, file.relative_to(repo))

    for file in outputs_dir.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(repo))

print(f"Created: {zip_path}")


Created: C:\Users\aamin\ThemeCloner2\ThemeCloner_V3_run_results.zip


In [22]:
# -------------------- 9. Git commit and push --------------------
# Save the notebook before running this cell.
import subprocess

FILES_TO_COMMIT = [
    "ThemeCloner_V2_WalkForward_revised.ipynb",
    "src/matching_v3.py",
    "src/walkforward_v3.py",
    "src/evaluation_v3.py",
]

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

run_git("status", "--short")
run_git("add", *FILES_TO_COMMIT)
run_git("diff", "--cached", "--stat")
run_git("commit", "-m", "Improve theme matching diagnostics and parallel placebos")
run_git("push")


 M ThemeCloner_V2_WalkForward.ipynb
 M ThemeCloner_V2_WalkForward_revised.ipynb
 M outputs/fig_v2_walkforward_equity.png
?? .ipynb_checkpoints/ThemeCloner_V2_WalkForward_revised-checkpoint.ipynb
?? IMPLEMENTATION_NOTES.md
?? ThemeCloner_V3_run_results.zip
?? outputs/semantic_review_key_PRIVATE.csv
?? outputs/semantic_review_sheet.csv
?? outputs/v3_basket_history.csv
?? outputs/v3_candidate_forward_exposure.csv
?? outputs/v3_forward_evaluation.csv
?? outputs/v3_overlap_diagnostics.csv
?? outputs/v3_placebo_rank_thresholds.csv
?? outputs/v3_theme_reference_distances.csv
?? outputs/v3_timings.csv
?? src/__pycache__/evaluation_v3.cpython-310.pyc
?? src/__pycache__/matching_v3.cpython-310.pyc
?? src/__pycache__/walkforward_v3.cpython-310.pyc


 ThemeCloner_V2_WalkForward_revised.ipynb | 2906 ++----------------------------
 1 file changed, 102 insertions(+), 2804 deletions(-)

[master 6ffae3e] Improve theme matching diagnostics and parallel placebos
 1 file changed, 102 insertions(+), 2804 d

CompletedProcess(args=['git', 'push'], returncode=0, stdout='', stderr='To https://github.com/AA-mini/ThemeCloner2.git\n   d7da6c1..6ffae3e  master -> master\n')